# Comparacion: K-medoids + TSP  vs  Set Cover ILP

Carga los resultados de ambos enfoques y genera tabla comparativa y visualizacion lado a lado.

**Prerequisito**: haber ejecutado `data_generation.py`, `clustering.py`, `tsp_solver.py` (rama main) y `setcover_rutas.ipynb` (rama mariana).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import contextily as ctx

## 1. Carga de resultados

In [ ]:
scenario = np.load("scenario_data.npz")
x = scenario["x"]
y = scenario["y"]
origin_idx = int(scenario["origin_index"])
bus_capacity = int(scenario["bus_capacity"])
n_points = scenario["dist_matrix"].shape[0]
children_idx = [i for i in range(n_points) if i != origin_idx]
n_children = len(children_idx)

# K-medoids + TSP
clust = np.load("clustering_result.npz")
tsp   = np.load("tsp_result.npz")

# Set Cover ILP
sc = np.load("setcover_result.npz")

print(f"Estudiantes: {n_children}  |  Capacidad: {bus_capacity}  |  Min. teorico: {int(np.ceil(n_children/bus_capacity))} buses")

## 2. Tabla comparativa de metricas

In [ ]:
km_labels    = clust["labels"]
km_k         = int(clust["k"])
km_sizes     = [int(np.sum(km_labels == c)) for c in range(km_k)]
km_total     = float(tsp["total_distance"])
dist_matrix  = scenario["dist_matrix"]
tsp_dist     = np.array([
    sum(dist_matrix[tsp[f"route_{i}"][j], tsp[f"route_{i}"][j + 1]]
        for j in range(len(tsp[f"route_{i}"]) - 1))
    for i in range(km_k)
])

sc_routes    = sc["routes"]
sc_n_buses   = int(sc["n_buses"])
sc_dist      = sc["route_distances_m"]
sc_total     = float(sc["total_distance_m"])
sc_sizes     = [int(np.sum(row >= 0)) for row in sc_routes]

print("="*60)
print(f"{'Metrica':<35} {'K-med+TSP':>10} {'SetCover':>10}")
print("-"*60)
print(f"{'Buses usados':<35} {km_k:>10} {sc_n_buses:>10}")
print(f"{'Distancia total (km)':<35} {km_total/1000:>10.2f} {sc_total/1000:>10.2f}")
print(f"{'Dist. media por bus (km)':<35} {np.mean(tsp_dist)/1000:>10.2f} {np.mean(sc_dist)/1000:>10.2f}")
print(f"{'Tamano min. ruta':<35} {min(km_sizes):>10} {min(sc_sizes):>10}")
print(f"{'Tamano max. ruta':<35} {max(km_sizes):>10} {max(sc_sizes):>10}")
print(f"{'Tamano promedio':<35} {np.mean(km_sizes):>10.1f} {np.mean(sc_sizes):>10.1f}")
print("="*60)

## 3. Distribucion del tamano de rutas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for ax, sizes, title, color in zip(
    axes,
    [km_sizes, sc_sizes],
    ["K-medoids + TSP", "Set Cover ILP"],
    ["steelblue", "darkorange"],
):
    ax.bar(range(len(sizes)), sorted(sizes, reverse=True), color=color, edgecolor="white")
    ax.axhline(bus_capacity, ls="--", color="red", lw=1.2, label=f"Capacidad ({bus_capacity})")
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Bus (ordenado por tamano)")
    ax.set_ylabel("Estudiantes")
    ax.legend()

fig.suptitle("Tamano de rutas por enfoque", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig("comparacion_sizes.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Visualizacion lado a lado

In [ ]:
def distinct_colors(n):
    hues = np.linspace(0, 1, n, endpoint=False)
    return [mcolors.hsv_to_rgb((h, 0.75, 0.85)) for h in hues]


fig, axes = plt.subplots(1, 2, figsize=(22, 11))

ax = axes[0]
colors = distinct_colors(km_k)
for c in range(km_k):
    mask = km_labels == c
    ci = [children_idx[i] for i in range(n_children) if mask[i]]
    ax.scatter([x[i] for i in ci], [y[i] for i in ci],
               color=colors[c], s=18, zorder=3)
ax.scatter(x[origin_idx], y[origin_idx], marker="*", color="red",
           s=250, zorder=6, label="Colegio")
ax.set_title(f"K-medoids + TSP  ({km_k} buses, {km_total/1000:.1f} km)", fontsize=13)
ax.set_aspect("equal")
ax.legend(fontsize=9)
try:
    ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik,
                    zoom=14, crs="EPSG:4326")
except Exception:
    pass

ax = axes[1]
colors = distinct_colors(sc_n_buses)
for i, row in enumerate(sc_routes):
    members = [int(v) for v in row if v >= 0]
    ax.scatter([x[j] for j in members], [y[j] for j in members],
               color=colors[i], s=18, zorder=3)
ax.scatter(x[origin_idx], y[origin_idx], marker="*", color="red",
           s=250, zorder=6, label="Colegio")
ax.set_title(f"Set Cover ILP  ({sc_n_buses} buses, {sc_total/1000:.1f} km)", fontsize=13)
ax.set_aspect("equal")
ax.legend(fontsize=9)
try:
    ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik,
                    zoom=14, crs="EPSG:4326")
except Exception:
    pass

fig.suptitle("Comparacion de enfoques", fontsize=15, fontweight="bold")
plt.tight_layout()
fig.savefig("comparacion_mapa.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: comparacion_sizes.png, comparacion_mapa.png")